<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-architect-program/blob/main/Day_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Building a Document Summarizer with Claude

### Structured JSON Output with XML Tags

**What is Document Summarization?**

Imagine you're a manager at Razorpay reviewing 50 payment dispute documents a day. Each one is 5 pages long. By hand? Impossible. That's where AI summarization comes in—Claude reads the full document and gives you the essential facts in 30 seconds, formatted as clean JSON that your software can understand.

In this, you'll learn:
- Why AI summarization matters for your business
- How to write prompts that Claude understands
- How to get clean, usable JSON output every time
- Real-world examples from Zepto, Infosys, and Apollo Hospitals
- How to move from experimentation to production code

---

---

## Part 1️⃣: Why Summarization Matters (10 min)

### The Real-World Problem

You work at one of these companies:

- **Razorpay**: Payment disputes pile up. Each one is a 4-page PDF. 50 come in daily.
- **Zepto**: Supplier contracts. Need to extract payment terms, delivery requirements, risks from 100+ documents.
- **Apollo Hospitals**: Patient discharge notes (500 words each) need to be summarized for insurance claims (50 words max).
- **Infosys**: Client emails about project issues. Triage them by urgency: 🔴 Critical vs 🟡 Important vs 🟢 Low.
- **Nykaa**: Product review feedback (10,000+ per day). Extract what customers loved vs hated.

**The Challenge**: Reading 100 documents manually takes 8 hours. AI can do it in 2 minutes.

### Why JSON? Why Structure?

**Bad Summary** (unstructured text):
```
The customer says they are unhappy with the payment feature because it crashed
on Tuesday and they lost a transaction. They want a refund and an explanation.
Also they mentioned the interface is confusing.
```

**Good Summary** (structured JSON):
```json
{
  "issue_type": "Technical - Payment Crash",
  "customer_sentiment": "frustrated",
  "impact": "transaction_loss",
  "requested_action": "refund + explanation",
  "secondary_issues": ["UI/UX confusion"],
  "urgency": "high",
  "owner_team": "payments_support"
}
```

**Why the JSON version is better:**
- 🤖 **Software can read it**: Your database can ingest it automatically.
- 📊 **Searchable**: Find all `urgency: high` issues in milliseconds.
- 🔍 **No ambiguity**: `customer_sentiment: frustrated` is clear (vs. tone guessing).
- 📈 **Scalable**: Process 1,000 documents without manual reading.

### Quick Recap

✅ **Do this**: Ask Claude for JSON output with clear field names.

❌ **Don't do this**: Accept unstructured text summaries; parse them later.

### Try This Yourself

Think of a document *you* read regularly (contract, email, report). What 3–5 fields would make a good JSON summary? Example: If it's a contract, fields might be: `{duration, cost_per_month, termination_clause, renewal_date}`

---

## Part 2️⃣: Understanding Summarization Types

### Extractive Summarization

**What it is**: Copy the most important sentences *directly from the original document*. Don't rewrite.

**Original document:**
```
Zepto signed a contract with Supplier X on March 1, 2025. Payment terms are net-30.
Supplier X must deliver within 48 hours or faces a 5% penalty. The contract renews annually
unless terminated with 90 days' notice. Supplier X has never missed a delivery in 5 years.
```

**Extractive summary:**
```json
{
  "key_facts": [
    "Payment terms are net-30.",
    "Supplier X must deliver within 48 hours or faces a 5% penalty.",
    "The contract renews annually unless terminated with 90 days' notice."
  ],
  "method": "sentences copied directly from source"
}
```

**When to use**: Legal compliance, audit trails, or when you need to quote original language.

### Abstractive Summarization

**What it is**: Claude **rewrites the content in its own words**. More concise, human-friendly, but NOT quoted from the source.

**Same original document:**
```
Zepto signed a contract with Supplier X on March 1, 2025. Payment terms are net-30.
Supplier X must deliver within 48 hours or faces a 5% penalty. The contract renews annually
unless terminated with 90 days' notice. Supplier X has never missed a delivery in 5 years.
```

**Abstractive summary:**
```json
{
  "summary": "Zepto contracted Supplier X with 48-hour delivery SLA and net-30 payments. Late delivery incurs 5% penalty. Annual auto-renewal with 90-day cancellation window. Supplier has perfect 5-year track record.",
  "reliability_score": 0.95,
  "method": "rewritten for clarity"
}
```

**When to use**: Executive briefings, customer emails, meeting notes—anywhere humans need quick understanding.

### Comparison: Extractive vs Abstractive vs Hybrid

| Feature | Extractive | Abstractive | Hybrid |
|---------|-----------|-----------|--------|
| **Method** | Copy sentences from original | Rewrite in Claude's words | Mix both approaches |
| **Accuracy** | High (legally defensible) | Medium (Claude may infer) | High |
| **Conciseness** | Medium (full sentences) | High (Claude condenses) | High |
| **Best for** | Legal, compliance | Business briefings | Contracts + summaries |
| **Example use case** | Patent review | Email triage | Supplier contract analysis |

### ⚡ Important: Claude Does Abstractive by Default

When you ask Claude to summarize, it naturally rewrites content. To force extractive output, you must explicitly say: `"Extract the 3 most important sentences directly from the document without rewording."` Otherwise, Claude will abstract.

### Quick Recap

✅ **Use Extractive** when: Legal compliance, audit trails, quoting source.

✅ **Use Abstractive** when: Speed, executive briefing, human readability.

✅ **Use Hybrid** when: You need both security (extracted facts) and clarity (rewritten context).

### Try This Yourself

Take a 300-word article about Infosys' latest earnings. Extract 2 sentences. Rewrite the same content in 1 sentence. Which is clearer?

---

## Part 3️⃣: XML Tags for Clarity

### What Are XML Tags and Why Claude Loves Them

XML tags are **labels that tell Claude what kind of information is inside**. Think of them like folders in your filing cabinet.

```xml
<folder_name>contents go here</folder_name>
```

Claude was trained on **code and structured data** (XML, JSON, HTML), so it naturally understands tags. They act like **visual separators** that prevent confusion.

### Common Tags for Summarization

| Tag | Purpose | Example |
|-----|---------|---------|
| `<instructions>` | What you want Claude to do | `<instructions>Summarize this contract</instructions>` |
| `<document>` | The text to summarize | `<document>Full contract text here...</document>` |
| `<output_format>` | Exact JSON structure expected | `<output_format>{"key": "value"}</output_format>` |
| `<example>` | Sample input + output | `<example>Input doc → Output JSON</example>` |
| `<rules>` | Constraints and guidelines | `<rules>Max 200 words, detect tone, extract names</rules>` |

### 🔴 Bad Prompt (Confusing)

```
Summarize this Razorpay dispute document. The customer says the payment feature crashed
on Tuesday and they lost ₹5,000. They want a refund. Extract the issue type, sentiment,
and urgency. Return JSON. Here's a payment dispute about a failed transaction on the
Razorpay platform where the customer lost access to their account and wants ₹5,000
refunded. Tell me the customer name, the amount, and what they want done about it.
```

**Problems:**
- ❌ Instruction mixed with document text (confusing where one ends, one begins)
- ❌ Multiple asks without clear structure
- ❌ No JSON example provided
- ❌ Claude gets confused about what's the prompt vs. the document

### 🟢 Good Prompt (Clear Tags)

```
<instructions>
Summarize this Razorpay dispute document. Extract structured information in JSON.
</instructions>

<document>
The customer says the payment feature crashed on Tuesday and they lost ₹5,000.
They want a refund. They cannot access their account anymore.
</document>

<output_format>
{
  "customer_name": "extracted from doc",
  "amount_affected": "in rupees",
  "issue_type": "payment_crash OR account_access OR other",
  "requested_action": "refund OR investigation OR other",
  "urgency": "critical OR high OR medium"
}
</output_format>
```

**Why it's better:**
- ✅ Instructions separated from document (no ambiguity)
- ✅ One clear task per section
- ✅ JSON template provided (Claude knows exact structure expected)
- ✅ Claude outputs clean, predictable JSON

### Rule: One Tag Per Concept (Don't Nest Deep)

**❌ Too nested** (confusing):
```xml
<prompt>
  <task>
    <instruction>
      Summarize this document
    </instruction>
  </task>
</prompt>
```

**✅ Flat and clear** (preferred):
```xml
<instructions>Summarize this document</instructions>
<document>...</document>
<output_format>...</output_format>
```

### Quick Recap

✅ **Do this**: Use tags like `<instructions>`, `<document>`, `<output_format>` to separate concerns.

❌ **Don't do this**: Mix instructions and document text; don't nest tags deeply.

### Try This Yourself

Write a prompt about summarizing an IIT Bombay research paper using proper XML tags.

---

## Part 4️⃣: Building the Perfect Summarizer Prompt

### The 5-Ingredient Prompt Formula

Every great prompt has 5 layers, in this order:

| # | Layer | Purpose | Example |
|---|-------|---------|---------|
| 1️⃣ | **Role** | Who is Claude for this task? | "You are a legal contract analyst" |
| 2️⃣ | **Task** | What exactly do you want? | "Summarize this supplier contract" |
| 3️⃣ | **Rules** | Constraints on the output | "Extract only explicit facts; don't infer" |
| 4️⃣ | **Format** | Exact JSON structure | `{"key1": "type", "key2": "type"}` |
| 5️⃣ | **Example** | Sample input + output | Show a real example so Claude mimics it |

### 🔴 WEAK Prompt Example (Customer Support Ticket)

```
Summarize this customer support ticket:

Customer: I have a problem with my Zepto order. It never arrived. I paid ₹499.
When I called support, they hung up on me. I'm very upset. I want my money back
or a new delivery. Also the app is buggy on my phone.

Return JSON.
```

**What's wrong:**
- ❌ No role specified (Claude doesn't know it's a support analyst)
- ❌ "Return JSON" with no template (Claude guesses at structure)
- ❌ No rules about what facts to extract (vague)
- ❌ No example output (Claude invents field names)

**Claude's messy output:**
```json
{
  "issue": "order_issue",
  "description": "They want money back or redelivery",
  "angry": true
}
```

**Problems with this output:**
- Missing customer name, order ID, exact amount
- Field names are inconsistent (will break your database)
- No urgency level, no owner assignment

### 🟢 STRONG Prompt (Same Ticket, Better Structured)

```
<role>
You are a Zepto customer support analyst. Your job is to triage issues and assign them
to the right team. Your output will be stored in a database and used to measure performance.
</role>

<task>
Summarize this customer support ticket and extract structured data.
</task>

<rules>
1. Extract ONLY facts stated by the customer. Do not infer.
2. Sentiment: Choose from [satisfied, neutral, frustrated, angry]. If they say 'very upset', use 'angry'.
3. Urgency: [low, medium, high, critical]. No refund = critical.
4. Max 150 words for summary field.
5. If customer name is not mentioned, use 'unknown'.
6. Confidence: 1.0 = certain, 0.5 = ambiguous, 0.0 = unsure.
</rules>

<output_format>
{
  "customer_name": "string",
  "order_id": "string (if mentioned)",
  "amount_affected_inr": "number",
  "primary_issue": "order_delayed OR order_not_arrived OR support_rudeness OR app_bug OR other",
  "secondary_issues": ["array of strings"],
  "customer_sentiment": "satisfied OR neutral OR frustrated OR angry",
  "urgency": "low OR medium OR high OR critical",
  "requested_action": "refund OR redelivery OR explanation OR investigation",
  "confidence_score": "0.0 to 1.0",
  "suggested_owner_team": "logistics OR support OR app_team OR finance",
  "summary": "2-3 sentence summary of the issue"
}
</output_format>

<example>
INPUT: Customer says they ordered samosas on Tuesday, paid ₹120, delivery was scheduled for Wednesday morning
but the order never arrived. Customer got a refund automatically after 24 hours.

OUTPUT:
{
  "customer_name": "unknown",
  "order_id": "unknown",
  "amount_affected_inr": 120,
  "primary_issue": "order_not_arrived",
  "secondary_issues": [],
  "customer_sentiment": "neutral",
  "urgency": "low",
  "requested_action": "none",
  "confidence_score": 0.95,
  "suggested_owner_team": "logistics",
  "summary": "Samosa order of ₹120 scheduled for Wednesday never arrived. Auto-refunded after 24 hours. No customer response."
}
</example>
```

### Claude's Output (Clean & Structured)

```json
{
  "customer_name": "unknown",
  "order_id": "unknown",
  "amount_affected_inr": 499,
  "primary_issue": "order_not_arrived",
  "secondary_issues": ["support_rudeness", "app_bug"],
  "customer_sentiment": "angry",
  "urgency": "critical",
  "requested_action": "refund",
  "confidence_score": 0.88,
  "suggested_owner_team": "logistics",
  "summary": "Customer's ₹499 order never arrived. Customer support hung up during call. Demanding refund or redelivery. Zepto app also buggy on customer's phone."
}
```

**Why this is better:**
- ✅ All fields present (consistency for database storage)
- ✅ Clear sentiment and urgency (for routing)
- ✅ Confidence score shows Claude's certainty (0.88 = high confidence, all facts stated clearly)
- ✅ `suggested_owner_team` automates routing to the right department

### The Prompt Template (Copy-Paste Ready)

Use this template for ANY summarization task:

```
<role>
You are a [your_role]. Your output will be [what happens to it].
</role>

<task>
[What exactly do you want Claude to do?]
</task>

<rules>
1. [Rule 1]
2. [Rule 2]
3. [Rule 3 about output structure]
</rules>

<output_format>
{
  "field1": "type (string, number, boolean, array)",
  "field2": "type"
}
</output_format>

<example>
INPUT: [sample input]
OUTPUT: [sample JSON output]
</example>

<document>
[Actual document to summarize goes here]
</document>
```

### Quick Recap

✅ **Always include**: Role, Task, Rules, Format, Example (in that order).

❌ **Never skip**: The example. Examples teach Claude the style you want.

### Try This Yourself

Take the weak prompt from earlier. Rewrite it using the 5-ingredient formula.

---

## Part 5️⃣: Advanced Features

### Feature 1️⃣: Confidence Scoring

**What**: Tell Claude to assign a confidence score (0.0 = 'I don't know', 1.0 = 'certain') to each fact extracted.

**Why**: Prevents hallucination. If Claude is 30% confident, that's a red flag.

**Example: Apollo Hospitals Medical Report**

```
<rules>
For each extracted fact, assign a confidence_score:
- 1.0 = explicitly stated in document
- 0.7 = clearly implied, but not stated word-for-word
- 0.3 = uncertain, requires interpretation
- 0.0 = not mentioned in document at all

NEVER make up facts. If unsure, say 'cannot determine' and use confidence 0.0.
</rules>
```

**Sample output:**
```json
{
  "patient_diagnosis": {"value": "Type 2 Diabetes", "confidence": 1.0},
  "medication_dosage": {"value": "10mg daily", "confidence": 0.95},
  "life_expectancy_impact": {"value": "cannot determine", "confidence": 0.0},
  "recommended_follow_up": {"value": "unclear; doctor to decide", "confidence": 0.4}
}
```

### Feature 2️⃣: Tone Detection

**What**: Extract the customer's emotional tone from text.

**Why**: Routes urgent issues to the right team (angry customers need escalation).

**Example: Nykaa Customer Feedback**

**Sample 1 (Normal tone):**
```
Customer: "The moisturizer is good. It hydrates my skin. I'll order again."
Tone: positive
Urgency: low
Action: thank customer, add to loyalty rewards
```

**Sample 2 (Frustrated tone):**
```
Customer: "This cream DOESN'T WORK. My skin got worse. I want a refund NOW!!!"
Tone: angry
Urgency: critical
Action: escalate to customer success, immediate refund approval
```

**Sample 3 (Urgent tone):**
```
Customer: "Product arrived damaged. Need replacement for tomorrow's event."
Tone: urgent (not angry, just time-sensitive)
Urgency: high
Action: overnight shipping + apology discount
```

**How to prompt for this:**
```
Detect tone: positive OR neutral OR frustrated OR angry OR urgent
Rules:
- ALL CAPS or multiple exclamation marks = angry or urgent
- Words like 'immediately', 'today', 'tomorrow' = urgent
- Compliments = positive
```

### Feature 3️⃣: Action Items Extraction

**What**: Pull out 'to-do' items from meeting notes or emails, with owner, deadline, and dependencies.

**Why**: Automates task assignment. No more "who's doing what?" confusion.

**Real Example: Apollo Hospitals Meeting Notes**

**Meeting notes input:**
```
Dr. Singh said the patient imaging results are ready. Dr. Patel needs to review them
by Friday to decide on surgery. We're waiting on the pathology report from the lab
(should arrive Tuesday). If it's positive, Dr. Patel will call the patient on Friday.
Finance needs to approve the surgery cost before Wednesday for scheduling.
```

**Action items JSON:**
```json
{
  "action_items": [
    {
      "action": "Review patient imaging results",
      "owner": "Dr. Patel",
      "deadline": "Friday",
      "priority": "high",
      "blocked_by": ["Pathology report (arrives Tuesday)"]
    },
    {
      "action": "Call patient with surgical decision",
      "owner": "Dr. Patel",
      "deadline": "Friday",
      "priority": "high",
      "blocked_by": ["Review imaging results"]
    },
    {
      "action": "Approve surgery cost",
      "owner": "Finance Team",
      "deadline": "Wednesday",
      "priority": "high",
      "blocked_by": []
    }
  ]
}
```

### Feature 4️⃣: Edge Cases (Handle Gracefully)

**Edge Case 1: Empty Document**
```
INPUT: [blank document]

OUTPUT:
{
  "status": "no_content",
  "message": "No document provided for summarization",
  "summary": null,
  "confidence_score": 0.0
}
```

**Edge Case 2: Unrelated Content**
```
INPUT: "The quick brown fox jumps over the lazy dog."
CONTEXT: Supposed to be a Zepto supplier contract.

OUTPUT:
{
  "status": "invalid_document",
  "message": "Document does not appear to be a supplier contract. Cannot extract expected fields.",
  "confidence_score": 0.05
}
```

**Edge Case 3: Ambiguous Section**
```
INPUT: "It may or may not be possible to extend the contract."
EXTRACTION: {"renewal_terms": "unclear", "confidence": 0.2}
```

**How to prompt for edge cases:**
```
<rules>
If document is empty, return {status: 'no_content', message: 'Please provide a document'}.
If document is unrelated to the task, return {status: 'invalid_document', message: '[reason]'}.
If a field is ambiguous, set confidence < 0.5 and note the ambiguity.
NEVER hallucinate. If unsure, say 'unknown' or 'cannot determine'.
</rules>
```

### What You CAN and CAN'T Reliably Extract

| Feature | Reliable? | Why | Confidence |
|---------|-----------|-----|------------|
| **Explicit facts** (dates, names, amounts) | ✅ Yes | Claude reads them directly | 0.9–1.0 |
| **Tone/sentiment** | ✅ Yes | Patterns in language are clear | 0.8–0.95 |
| **Action items** | ✅ Yes | Often explicit ("do X by Friday") | 0.85–0.95 |
| **Ambiguous intent** | ⚠️ Medium | Requires interpretation | 0.4–0.7 |
| **Future predictions** | ❌ No | Claude can only infer from text | 0.0–0.3 |
| **Hidden costs** | ⚠️ Medium | Only detectable if they're hinted at | 0.3–0.6 |
| **Legal risk** | ⚠️ Medium | Subjective; Claude may over/understate | 0.5–0.8 |

### Quick Recap

✅ **Add confidence scores** to show Claude's certainty.

✅ **Detect tone** to automate customer routing.

✅ **Extract action items** to automate task assignment.

✅ **Handle edge cases** (empty docs, unrelated content) gracefully.

### Try This Yourself

Write a prompt that extracts action items from an Infosys project status email. Include owner names, deadlines, and blocked_by dependencies.

---

## Part 6️⃣: Real-World Example — Zepto Supplier Contract

### The Business Problem

Zepto receives supplier agreements from 500+ vendors. Each contract is 8–12 pages. Zepto needs to extract:
- **Payment terms**: When does Zepto pay? (net-30, net-60, upfront?)
- **Delivery requirements**: What's the SLA? Late penalties?
- **Termination clauses**: How to exit the contract?
- **Risks**: Hidden costs, unfair terms, automatic renewals?

Reading each contract manually takes 30 minutes. With 20 new contracts per week, that's 10 hours of work. Solution: **One prompt, 2 seconds per contract.**

### The Full Production-Ready Prompt

```
<system_message>
You are a Zepto supply chain analyst. You review vendor agreements and extract
critical business terms. Your output is stored in Zepto's contract database
and used for vendor negotiation, compliance, and risk assessment.
</system_message>

<role>
Analyze supplier contracts with precision. Extract ONLY facts explicitly stated.
Do not infer or assume. Flag ambiguities with low confidence scores.
</role>

<task>
Extract key business terms from this supplier agreement and return structured JSON.
</task>

<rules>
1. PAYMENT TERMS: Extract exact duration (net-30, net-60, etc.). If not stated, set confidence 0.0.
2. DELIVERY SLA: What is the promised delivery window? (e.g., 48 hours, next business day)
3. LATE PENALTIES: Any financial penalty for late delivery? Extract percentage or amount.
4. TERMINATION: How many days' notice required to cancel? (e.g., 90 days, 6 months)
5. RENEWAL: Does the contract auto-renew? If yes, when and how to prevent it?
6. RISK FLAGS:
   - Automatic renewal without opt-out = flag as HIGH RISK
   - Unilateral amendment rights = flag as MEDIUM RISK
   - Ambiguous payment terms = flag as MEDIUM RISK
   - All other terms = LOW RISK
7. Max 50 words per field. If field is not mentioned, use 'not specified'.
8. Confidence: 1.0 = explicit, 0.7 = implied, 0.0 = not mentioned.
</rules>

<output_format>
{
  "vendor_name": "string",
  "contract_start_date": "YYYY-MM-DD or 'not specified'",
  "contract_end_date": "YYYY-MM-DD or 'not specified'",
  "payment_terms": {"value": "string", "confidence": 0.0–1.0},
  "delivery_sla": {"value": "string", "confidence": 0.0–1.0},
  "late_delivery_penalty": {"value": "string or amount", "confidence": 0.0–1.0},
  "termination_notice_days": {"value": "number or 'not specified'", "confidence": 0.0–1.0},
  "auto_renewal": {"value": "yes OR no OR unclear", "confidence": 0.0–1.0},
  "renewal_notice_required": {"value": "yes OR no OR unclear", "confidence": 0.0–1.0},
  "risk_flags": ["HIGH_RISK_auto_renewal", "MEDIUM_RISK_ambiguous_payment", etc],
  "overall_risk_level": "LOW OR MEDIUM OR HIGH",
  "executive_summary": "2–3 sentence plain-English summary of key terms",
  "recommended_action": "approve OR renegotiate OR escalate_legal"
}
</output_format>

<example>
INPUT CONTRACT EXCERPT:
---
SUPPLIER AGREEMENT
Vendor: FreshProduce Co.
Effective: 1 January 2025
Term: 2 years, auto-renews annually unless 60 days' written notice provided.
Payment: Net-30 from invoice date. 2% early payment discount if paid within 10 days.
Delivery: Zepto orders must be fulfilled within 48 hours of order placement.
Late Penalties: ₹500 per delayed shipment or 5% order value, whichever is greater.
Termination: Either party may terminate with 90 days' written notice.
---

OUTPUT:
{
  "vendor_name": "FreshProduce Co.",
  "contract_start_date": "2025-01-01",
  "contract_end_date": "2027-01-01",
  "payment_terms": {"value": "Net-30; 2% discount if paid within 10 days", "confidence": 1.0},
  "delivery_sla": {"value": "48 hours from order placement", "confidence": 1.0},
  "late_delivery_penalty": {"value": "₹500 per delay or 5% of order value (whichever is greater)", "confidence": 1.0},
  "termination_notice_days": {"value": 90, "confidence": 1.0},
  "auto_renewal": {"value": "yes", "confidence": 1.0},
  "renewal_notice_required": {"value": "yes, 60 days written notice", "confidence": 1.0},
  "risk_flags": ["HIGH_RISK_auto_renewal"],
  "overall_risk_level": "MEDIUM",
  "executive_summary": "FreshProduce Co. delivers within 48 hours with clear late penalties. Contract auto-renews; Zepto must provide 60-day notice to exit. Payment is net-30 with early payment incentive.",
  "recommended_action": "approve with calendar reminder for 60-day renewal notice"
}
</example>
```

### How to Use This in Practice

**Step 1**: Copy the prompt above.

**Step 2**: Replace `<example>` with a real contract excerpt from your vendor.

**Step 3**: Paste the actual supplier contract in a new section:
```
<document>
[Paste the full vendor agreement here]
</document>
```

**Step 4**: Send to Claude API or Claude.ai. Claude returns clean JSON.

**Step 5**: Parse the JSON and import into your contract database.

### Why This Prompt Works

✅ **Role is clear**: "You are a supply chain analyst" (Claude knows what hat to wear)

✅ **Output format is precise**: Every field is defined with type (string, number, etc.)

✅ **Example is realistic**: The sample output matches the template exactly (Claude copies the style)

✅ **Confidence scores prevent hallucination**: If Claude doesn't know, it says so

✅ **Risk flags automate escalation**: `HIGH_RISK_auto_renewal` triggers a calendar reminder

### Iterating If Claude Misses Something

**Problem**: Claude missed the "early payment discount" in the first run.

**Solution**: Add a rule:
```
<rules>
...
IMPORTANT: If vendor offers a discount for early payment, extract the discount
percentage and include it in the 'payment_terms' field.
Example: "Net-30, but 2% discount if paid within 10 days"
</rules>
```

**Rerun**: Claude now includes the discount.

### Quick Recap

✅ **Use this exact prompt** for any supplier contract.

✅ **Iterate on the rules** if Claude misses something.

✅ **Monitor confidence scores** to catch hallucinations.

### Try This Yourself

Modify this prompt to extract terms from an Infosys software license agreement instead. What fields would you add or remove?

---

## Part 7️⃣: From Prompt to Production

### Using Claude API (Brief Overview)

**What**: Instead of visiting Claude.ai in a browser, you call Claude programmatically with Python.

**Why**: Automate processing 1,000 contracts without clicking a button 1,000 times.

**Minimal example:**
```python
from anthropic import Anthropic

client = Anthropic()

response = client.messages.create(
    model="claude-opus-4-7",  # Latest Claude model (as of April 2026)
    max_tokens=1024,
    messages=[
        {"role": "user", "content": """
        <role>You are a contract analyst...</role>
        <document>[paste contract here]</document>
        [rest of your prompt]
        """}
    ]
)

# Claude's response
print(response.content[0].text)
```

For the full API documentation, see: https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/overview

In [2]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 11.4 MB/s eta 0:00:00


In [11]:
from anthropic import Anthropic
import json
import re

from google.colab import userdata
key = userdata.get('MY_API_KEY')

client = Anthropic(api_key=key)

system_prompt = """You are a Zepto supply chain analyst. You review vendor agreements and extract
critical business terms. Your output is stored in Zepto's contract database
and used for vendor negotiation, compliance, and risk assessment.

RULES:
1. Extract ONLY facts explicitly stated. Do not infer or assume.
2. PAYMENT TERMS: Extract exact duration (net-30, net-60, etc.). If not stated, confidence = 0.0.
3. DELIVERY SLA: Promised delivery window (e.g., 48 hours, next business day).
4. LATE PENALTIES: Financial penalty for late delivery — percentage or amount.
5. TERMINATION: Days' notice required to cancel.
6. RENEWAL: Does contract auto-renew? If yes, how to prevent it?
7. RISK FLAGS:
   - Automatic renewal without opt-out → HIGH_RISK_auto_renewal
   - Unilateral amendment rights       → MEDIUM_RISK_unilateral_amendment
   - Ambiguous payment terms           → MEDIUM_RISK_ambiguous_payment
   - All other unclear terms           → LOW_RISK
8. Max 50 words per field. If a field is not mentioned, use "not specified".
9. Confidence: 1.0 = explicit, 0.7 = implied, 0.0 = not mentioned.

IMPORTANT: Return ONLY a raw JSON object. No markdown, no code fences, no explanation."""

contract_text = """
SUPPLIER AGREEMENT

Vendor: RajAgro Fresh Pvt. Ltd.
Effective Date: 1 March 2025
Term: 3 years (ending 28 February 2028). Contract auto-renews for 1 year unless
either party provides 45 days written notice before expiry.

Payment: Net-45 from invoice date. A 1.5% late payment interest applies if payment
exceeds Net-45. No early payment discount offered.

Delivery: All Zepto purchase orders must be fulfilled and dispatched within 72 hours
of order confirmation. Cold-chain orders must be delivered within 36 hours.

Late Delivery Penalty: Rs. 1,000 per delayed shipment, or 3% of the delayed
order value, whichever is higher. Penalties waived for delays caused by
government-mandated restrictions or natural disasters.

Termination: Either party may terminate this agreement with 60 days written notice.
Zepto reserves the right to terminate immediately without notice in case of
repeated quality failures (3 or more incidents in a 90-day window).

Amendments: RajAgro Fresh reserves the right to amend pricing schedules with
15 days prior written notice to Zepto. All other amendments require mutual
written consent.

Governing Law: This agreement is governed by the laws of Maharashtra, India.
Disputes shall be resolved via arbitration in Mumbai.
"""

user_prompt = f"""Extract all key business terms from the supplier contract below and return a single JSON object.

<contract>
{contract_text.strip()}
</contract>

Return this exact JSON structure (no markdown, no prose — raw JSON only):
{{
  "vendor_name": "string",
  "contract_start_date": "YYYY-MM-DD or not specified",
  "contract_end_date": "YYYY-MM-DD or not specified",
  "payment_terms": {{"value": "string", "confidence": 0.0}},
  "delivery_sla": {{"value": "string", "confidence": 0.0}},
  "late_delivery_penalty": {{"value": "string", "confidence": 0.0}},
  "termination_notice_days": {{"value": "number or not specified", "confidence": 0.0}},
  "auto_renewal": {{"value": "yes or no or unclear", "confidence": 0.0}},
  "renewal_notice_required": {{"value": "yes or no or unclear", "confidence": 0.0}},
  "risk_flags": [],
  "overall_risk_level": "LOW or MEDIUM or HIGH",
  "executive_summary": "2-3 sentence plain-English summary",
  "recommended_action": "approve or renegotiate or escalate_legal"
}}"""

message = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=2048,
    system=system_prompt,
    messages=[
        {"role": "user", "content": user_prompt}
    ],
)

print(message.content[0].text)

def extract_json(text: str) -> dict:
    if not text or not text.strip():
        raise ValueError("Empty response from Claude")

    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass

    raise ValueError(f"No valid JSON found in response:\n{text[:500]}")

print("Stop reason  :", message.stop_reason)
print("Input tokens :", message.usage.input_tokens)
print("Output tokens:", message.usage.output_tokens)
print()

raw_text = message.content[0].text if message.content else ""
parsed = extract_json(raw_text)
print(json.dumps(parsed, indent=2))

```json
{
  "vendor_name": "RajAgro Fresh Pvt. Ltd.",
  "contract_start_date": "2025-03-01",
  "contract_end_date": "2028-02-28",
  "payment_terms": {"value": "Net-45 from invoice date. 1.5% late payment interest if payment exceeds Net-45.", "confidence": 1.0},
  "delivery_sla": {"value": "72 hours for standard orders; 36 hours for cold-chain orders from confirmation.", "confidence": 1.0},
  "late_delivery_penalty": {"value": "Rs. 1,000 per shipment or 3% of order value (whichever higher). Waived for govt restrictions or natural disasters.", "confidence": 1.0},
  "termination_notice_days": {"value": 60, "confidence": 1.0},
  "auto_renewal": {"value": "yes", "confidence": 1.0},
  "renewal_notice_required": {"value": "yes", "confidence": 1.0},
  "renewal_notice_days": 45,
  "risk_flags": ["HIGH_RISK_auto_renewal", "MEDIUM_RISK_unilateral_amendment"],
  "overall_risk_level": "MEDIUM",
  "executive_summary": "3-year agreement with auto-renewal requiring 45-day opt-out notice. RajAgro reser

### Error Handling: What If JSON Is Invalid?

**Option A (Recommended): Use Claude's Native Structured Outputs API**

As of late 2025, Anthropic released a **Structured Outputs** feature that *guarantees* schema-valid JSON — no parsing errors, no retries needed. This is now generally available for Claude Haiku 4.5, Sonnet 4.6, Opus 4.7, and other recent models.

```python
from anthropic import Anthropic
import json

client = Anthropic()

response = client.messages.create(
    model="claude-opus-4-7",
    max_tokens=1024,
    messages=[{"role": "user", "content": "[your prompt]"}],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string"},
                    "urgency": {"type": "string"}
                },
                "required": ["summary", "urgency"]
            }
        }
    }
)
# Response guaranteed to be valid JSON matching your schema
result = json.loads(response.content[0].text)
```

See: https://platform.claude.com/docs/en/build-with-claude/structured-outputs

**Option B (Fallback): Ask Claude to Self-Correct**

If you're using prompt-based JSON (without the structured outputs API) and parsing fails:

```python
import json

try:
    result = json.loads(claude_response)
except json.JSONDecodeError as e:
    # Ask Claude to fix the JSON
    fix_prompt = f"""
    The JSON I got from you has an error: {e}
    Please return ONLY valid JSON, nothing else.
    """
    # Rerun Claude with the fix_prompt
```

### Rate Limits: How Fast Can You Go?

Rate limits vary by your usage tier and the model you use. There is no single "50 per minute" number that applies to everyone.

| Tier | Who It's For | Limits |
|------|-------------|--------|
| **Tier 1** | New developers, small projects | Lower limits to start |
| **Tier 2** | Growing applications | Increased limits |
| **Tier 3–4** | Established applications | Higher limits |
| **Enterprise** | Large-scale production | Custom limits |

**Practical advice:**
- Check your specific rate limits in the Claude Console under Settings → Limits
- For processing thousands of documents, use the **Batch API** (async, 50% cheaper, higher throughput)
- Space out large jobs — don't send all 10,000 contracts at once

For current rate limit documentation: https://platform.claude.com/docs/en/api/rate-limits

### Testing: Quality Assurance Before Production

**Step 1**: Run your prompt on 10 sample documents.

**Step 2**: Manually review Claude's output. Ask:
- ✅ Are all fields filled correctly?
- ✅ Is confidence accurate? (High confidence on explicit facts? Low on ambiguous facts?)
- ✅ Did Claude hallucinate? (Make up facts not in the document?)
- ✅ Are JSON field names consistent?

**Step 3**: If Claude missed something, add a rule and retest.

**Step 4**: Once happy with results, deploy to production.

### Deployment: Where to Store Your Prompt

| Location | Best For | Why |
|----------|----------|-----|
| **Claude Projects** | Collaboration with team | Built-in version control, shared across team |
| **GitHub / GitLab** | Version tracking | Track prompt changes over time, revert if needed |
| **Environment variables** | Security | Don't hardcode sensitive prompts in code |
| **Database** | Dynamic prompts | Store different prompts for different document types |

**Recommended**: Start with Claude Projects, move to GitHub as it scales.

### Future: Claude Skills (Coming Soon)

Claude is evolving to support **Skills**—predefined tools for specific tasks:
- 📄 **File reading**: Upload PDFs, documents, and Claude processes them
- 💻 **Code execution**: Run Python to transform data
- 🔗 **Integrations**: Connect to databases, APIs, email

In the future, you might use a pre-built "Contract Analyzer" skill instead of writing the prompt from scratch.

**For now**: Master the prompting techniques in this lecture, and you'll be ready for Skills when they arrive.

### Quick Recap

✅ **Use Claude API** to automate large batches of documents.

✅ **Handle errors gracefully**: Malformed JSON? Ask Claude to fix it.

✅ **Plan for scale**: Rate limits, costs, testing before production.

✅ **Store prompts safely**: GitHub or Claude Projects (not hardcoded).

### Try This Yourself

Find a contract from IIT Bombay, Razorpay, or any company. Run it through the prompt above and see what Claude extracts.

---

## 🎯 Quick Recap: All 7 Parts in One Page

| Part | Key Takeaway | Example |
|------|--------------|---------|
| 1️⃣ **Why Summarization** | AI saves time; structured JSON is essential | Zepto contracts: 30 min → 2 sec per contract |
| 2️⃣ **Types of Summarization** | Extractive (copy text) vs Abstractive (rewrite); Claude does abstractive by default | Supplier contract: extract payment terms, or rewrite as briefing |
| 3️⃣ **XML Tags** | Structure prompts with `<role>`, `<task>`, `<document>`, `<output_format>` | `<instructions>Summarize</instructions><document>...</document>` |
| 4️⃣ **5-Ingredient Formula** | Role + Task + Rules + Format + Example | Every great prompt has all 5 layers |
| 5️⃣ **Advanced Features** | Add confidence scores, tone detection, action items, edge cases | Nykaa feedback: detect `tone: angry` → escalate |
| 6️⃣ **Real Example** | Copy-paste ready Zepto contract prompt | Extract payment terms, delivery SLA, risks |
| 7️⃣ **Production** | API, rate limits, costs, testing, storage | ₹100 to summarize 10,000 documents |

---

## 📋 Weak vs Strong Comparison (Quick Reference)

### Weak Prompt
```
Summarize this document and give me the key points.
```
❌ No role | ❌ No format | ❌ No example | ❌ Messy output

### Strong Prompt
```
<role>You are a [analyst role]. Your output will [where it's used].</role>
<task>Extract [specific facts] from this document.</task>
<rules>Rule 1... Rule 2... Rule 3...</rules>
<output_format>{"field1": "type", "field2": "type"}</output_format>
<example>INPUT: [sample] → OUTPUT: [sample JSON]</example>
<document>[actual document here]</document>
```
✅ Clear role | ✅ Exact format | ✅ Example provided | ✅ Clean output

---

## 🧳 Prompt Cheat Sheet: Copy-Paste Templates

### Template 1: General Document Summarizer

```
<role>You are a document analyst.</role>
<task>Summarize this document in JSON format.</task>
<rules>
1. Extract only explicit facts. Do not infer.
2. Max 150 words per field.
3. Assign confidence: 1.0 = certain, 0.0 = unsure.
</rules>
<output_format>
{
  "title": "string",
  "key_points": ["string"],
  "action_items": ["string"],
  "overall_confidence": 0.0–1.0
}
</output_format>
```

### Template 2: Contract Analyzer

```
<role>You are a legal contract specialist.</role>
<task>Extract business terms from this supplier contract.</task>
<rules>
1. Payment terms, delivery SLA, termination notice.
2. Flag high-risk clauses (auto-renewal, unilateral amendments).
3. Confidence: explicit facts = 1.0, implied = 0.7, unsure = 0.0.
</rules>
<output_format>
{
  "vendor": "string",
  "payment_terms": "string",
  "delivery_sla": "string",
  "termination_notice_days": "number",
  "risk_flags": ["string"],
  "overall_risk_level": "LOW OR MEDIUM OR HIGH"
}
</output_format>
```

### Template 3: Email Triage Bot

```
<role>You are a customer support triage specialist.</role>
<task>Analyze this customer email and route it to the right team.</task>
<rules>
1. Sentiment: satisfied, neutral, frustrated, angry.
2. Urgency: low, medium, high, critical.
3. Owner team: support, logistics, finance, technical, escalation.
</rules>
<output_format>
{
  "customer_sentiment": "string",
  "urgency": "string",
  "issue_type": "string",
  "suggested_owner_team": "string",
  "summary": "string"
}
</output_format>
```

### Template 4: Meeting Notes Processor

```
<role>You are a project manager extracting action items from meeting notes.</role>
<task>Extract all action items, owners, and deadlines.</task>
<rules>
1. Action: what needs to be done?
2. Owner: who's responsible?
3. Deadline: when?
4. Blocked by: what dependencies exist?
5. Priority: high, medium, low.
</rules>
<output_format>
{
  "action_items": [
    {"action": "string", "owner": "string", "deadline": "string", "priority": "string", "blocked_by": ["string"]}
  ]
}
</output_format>
```

---

## 🧪 Try This Yourself: 3 Practice Documents

### Practice 1: Customer Email

**Document:**
```
Hi support, I ordered a iPhone case 5 days ago. It still hasn't arrived even though
the website said 2–3 days delivery. I need it for my office. Can you send it faster
or cancel my order and refund me? I'm pretty frustrated. Order ID: #12345.
```

**Your task**: Run this through the Email Triage Bot template. Expected output: `sentiment: frustrated`, `urgency: high`, `owner_team: logistics`.

### Practice 2: Contract Excerpt

**Document:**
```
SUPPLIER AGREEMENT (Draft)
Company: TechSupply Inc.
Effective Date: April 1, 2025
Payment: Invoiced net-45 days from delivery.
Delivery: Standard shipping 5–7 business days. Express shipping available.
Termination: Either party may terminate with 30 days' notice.
Renewal: Agreement renews automatically for 1 year unless notice given 60 days prior.
```

**Your task**: Extract payment terms, delivery SLA, termination notice, auto-renewal. Expected confidence: mostly 1.0 (all explicit).

### Practice 3: Meeting Notes

**Document:**
```
Apollo Hospitals Weekly Sync
Dr. Singh said radiology reports are ready. Dr. Patel should review by Friday.
Finance needs to approve the imaging budget by Wednesday to process payments.
We're waiting on lab results from Tuesday; can't proceed until we have them.
Someone needs to send a follow-up email to the patient by Friday evening.
```

**Your task**: Extract action items (action, owner, deadline, blocked_by). Who's doing what by when?

---

## ⚠️ Common Mistakes Beginners Make (and How to Fix Them)

### Mistake 1: No JSON Example

**You write**: "Return JSON with key points."

**Claude outputs**: Inconsistent field names, missing fields.

**Fix**: Always include a `<example>` with sample input → sample JSON. Claude copies the style.

### Mistake 2: Asking Claude to Infer or Guess

**You write**: "What's the hidden cost in this contract?"

**Claude hallucinates**: Invents risks not mentioned in the contract.

**Fix**: Be explicit: "Extract ONLY facts stated in the document. If unsure, set confidence 0.0 and note 'not specified'." Add a rule: "NEVER infer or guess."

### Mistake 3: Mixing Instructions with Document Text

**You write**: "Summarize this contract about payment terms..." [then paste the whole contract inline]

**Claude gets confused**: Can't tell where the instruction ends and the document begins.

**Fix**: Use XML tags: `<instructions>...</instructions><document>...</document>`. Separate clearly.

### Mistake 4: Forgetting to Ask for Confidence Scores

**You write**: No mention of confidence.

**Result**: Claude outputs facts with 100% confidence even when uncertain, risking hallucination.

**Fix**: Always ask: "For each fact, assign confidence: 1.0 = explicit, 0.7 = implied, 0.0 = unsure."

### Mistake 5: Vague Sentiment/Urgency Rules

**You write**: "Detect sentiment: positive, negative, or neutral."

**Result**: Claude's definitions don't match yours. Frustrated customer marked as "negative" (vague).

**Fix**: Be specific: "positive = satisfied, neutral = no opinion, frustrated = unhappy but calm, angry = all-caps or threats."

---

## 📚 Further Reading & Resources

**Official Anthropic Resources:**
- Prompt Engineering Guide: https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/overview
- Structured Outputs (native JSON schema enforcement): https://platform.claude.com/docs/en/build-with-claude/structured-outputs
- Claude API Quickstart: https://platform.claude.com/docs/en/get-started
- Models & Pricing: https://platform.claude.com/docs/en/about-claude/models/overview

**Explore More:**
- Try Claude at https://claude.ai
- Claude API Pricing: https://platform.claude.com/docs/en/about-claude/pricing
- Anthropic Blog (latest releases): https://www.anthropic.com/news

**Next Steps:**
- Build a summarizer for your company's documents
- Scale it with the Claude API
- Share your prompts with teammates

---

## 🚀 Closing

### You Now Have Everything to Build a Production Summarizer

**You've learned:**
- ✅ Why AI summarization matters (saves time, costs, scales infinitely)
- ✅ How to structure prompts with XML tags (clear, unambiguous)
- ✅ The 5-ingredient formula (role, task, rules, format, example)
- ✅ How to add advanced features (confidence, tone, action items)
- ✅ How to move from Claude.ai to production with the API

### Next: Go Build

1. **Pick a document type** you work with daily (contracts, emails, reports, feedback).
2. **Write a prompt** using the 5-ingredient formula.
3. **Test with 10 samples** and check Claude's output.
4. **Iterate** if Claude misses something.
5. **Deploy** with Claude API once you're happy.

**You're going to build something great. Now go! 🚀**